# 02 — Analyse saisonnale

**Question centrale :** Le maïs a-t-il des patterns saisonniers exploitables ?

## Pourquoi c'est fondamental

Le maïs est une culture annuelle aux États-Unis. Son calendrier structure entièrement le marché :

| Période | Événement | Impact prix |
|---|---|---|
| Mars-Avril | Intentions de semis (Prospective Plantings) | Fort : révèle les intentions des agriculteurs |
| Avril-Mai | Plantations | Incertitude météo → prime de risque |
| Juillet | Pollinisation (période critique) | Très fort : la chaleur/sécheresse ici = catastrophe |
| Août-Sept | Remplissage des grains | Fort : pluie nécessaire |
| Sept-Oct | Récolte | Prix baissent = pression de vente |
| Nov-Déc | Exportations | Fenêtre d'export active |
| Jan-Fév | Bilan mi-année | Rapport WASDE revisé |

**Hypothèse :** Si ces patterns sont stables sur 25 ans, on peut les utiliser comme facteur de base.

In [ ]:
import sys
sys.path.insert(0, '../../src')
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

ROOT = Path('../../')
plt.rcParams['figure.figsize'] = (14, 6)
plt.style.use('seaborn-v0_8-whitegrid')

feat = pd.read_parquet(ROOT / 'data/processed/features.parquet')
tgt  = pd.read_parquet(ROOT / 'data/processed/targets.parquet')
feat['Date'] = pd.to_datetime(feat['Date'])
tgt['Date']  = pd.to_datetime(tgt['Date'])
df = feat.merge(tgt, on='Date', how='inner').sort_values('Date').reset_index(drop=True)

df['month']  = df['Date'].dt.month
df['year']   = df['Date'].dt.year
df['week']   = df['Date'].dt.isocalendar().week.astype(int)

# Trouver la colonne du prix du maïs
price_col = next((c for c in df.columns if 'corn_close' in c.lower() or 'corn_price' in c.lower()), None)
if price_col is None:
    price_col = next((c for c in df.columns if 'close' in c.lower() and 'corn' in c.lower()), 'corn_close')

print(f'Colonne prix utilisée : {price_col}')
print(f'Période : {df["Date"].min().date()} → {df["Date"].max().date()}')

## 1. Distribution des prix par mois

La première question : les prix sont-ils systématiquement différents selon le mois ?

In [ ]:
mois_fr = {1:'Jan',2:'Fév',3:'Mar',4:'Avr',5:'Mai',6:'Jun',7:'Jul',8:'Aoû',9:'Sep',10:'Oct',11:'Nov',12:'Déc'}

if price_col in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Distribution boxplot
    data_by_month = [df[df['month']==m][price_col].dropna().values for m in range(1,13)]
    bp = axes[0].boxplot(data_by_month, labels=[mois_fr[m] for m in range(1,13)], patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('steelblue')
        patch.set_alpha(0.6)
    axes[0].set_title('Distribution du prix du maïs par mois (2000-2025)')
    axes[0].set_ylabel('Prix CBOT (cts/bu)')
    
    # Rendement mensuel moyen
    df['ret_1m'] = df[price_col].pct_change(21)
    monthly_ret = df.groupby('month')['ret_1m'].agg(['mean','std']).reset_index()
    monthly_ret['month_name'] = monthly_ret['month'].map(mois_fr)
    
    axes[1].bar(monthly_ret['month_name'], monthly_ret['mean']*100,
                color=['green' if v>0 else 'red' for v in monthly_ret['mean']], alpha=0.7)
    axes[1].axhline(0, color='black', lw=0.8)
    axes[1].set_title('Rendement mensuel moyen sur 1 mois glissant (2000-2025)')
    axes[1].set_ylabel('Rendement moyen (%)')
    
    plt.tight_layout()
    plt.show()
else:
    print(f"Colonne {price_col} introuvable. Colonnes disponibles : {[c for c in df.columns if 'corn' in c.lower()][:10]}")

## 2. Patterns saisonniers par décennie

La saisonnalité est-elle stable dans le temps ? On compare 2000-2012 vs. 2013-2025.

In [ ]:
target_h20 = 'y_logret_h20'
if target_h20 not in df.columns:
    target_h20 = [c for c in df.columns if 'logret_h20' in c][0]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

periods = [
    ('2000-2025 (tout)', df),
    ('2000-2012 (pré-COT)', df[df['year'] <= 2012]),
    ('2013-2025 (avec COT)', df[df['year'] >= 2013]),
    ('2020-2025 (récent)', df[df['year'] >= 2020]),
]

for ax, (label, sub) in zip(axes.flat, periods):
    monthly = sub.groupby('month')[target_h20].agg(['mean','sem'])
    months = monthly.index
    means = monthly['mean'] * 100
    sems  = monthly['sem'] * 100
    
    ax.bar(months, means, color=['green' if v>0 else 'red' for v in means], alpha=0.6)
    ax.errorbar(months, means, yerr=sems*1.96, fmt='none', color='gray', capsize=4)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xticks(months)
    ax.set_xticklabels([mois_fr[m] for m in months])
    ax.set_title(f'Rendement moyen h=20j par mois — {label}')
    ax.set_ylabel('Rendement moyen (%) + IC 95%')

plt.suptitle('Saisonnalité des rendements h=20j — stabilité dans le temps', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 3. Heatmap : rendement par mois et par année

Cette vue révèle les années exceptionnelles (2012 = sécheresse, 2022 = Ukraine) et confirme si la saisonnalité est ou non un bruit.

In [ ]:
pivot = df.pivot_table(index='year', columns='month', values=target_h20, aggfunc='mean') * 100
pivot.columns = [mois_fr[m] for m in pivot.columns]

fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(pivot, cmap='RdYlGn', center=0, annot=True, fmt='.1f',
            linewidths=0.3, ax=ax, cbar_kws={'label': 'Rendement log h=20j (%)'})
ax.set_title('Rendement h=20j par mois et par année', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('Année')
plt.tight_layout()
plt.show()

## 4. Volatilité saisonnière

La volatilité (risque) est aussi saisonnière. En été = maximale (incertitude météo). En hiver = minimale.

In [ ]:
vol_monthly = df.groupby('month')[target_h20].std() * 100

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(vol_monthly.index, vol_monthly.values, color='orange', alpha=0.8)
ax.set_xticks(range(1,13))
ax.set_xticklabels([mois_fr[m] for m in range(1,13)])
ax.set_title('Volatilité saisonnière des rendements h=20j (écart-type par mois, 2000-2025)')
ax.set_ylabel('Écart-type (%)')

# Annotations calendrier
events = {4: 'Semis', 7: 'Pollini-\nsation', 9: 'Récolte', 12: 'Exports'}
for m, label in events.items():
    ax.annotate(label, xy=(m, vol_monthly[m]), xytext=(m, vol_monthly[m]+0.3),
                ha='center', fontsize=9, color='darkred')

plt.tight_layout()
plt.show()

## 5. Calendrier USDA vs. prix

Les publications USDA (WASDE = ~10 chaque année) créent des sauts de prix.
On visualise les rendements dans les 5 jours avant/après les dates WASDE.

In [ ]:
# Dates WASDE approximatives (10 ou 11 chaque année, autour du 10-12 du mois)
wasde_months = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

# Rendements à J-5, J-3, J-1, J0, J+1, J+3, J+5 autour du 10 de chaque mois
if price_col in df.columns:
    df['ret_1d'] = df[price_col].pct_change(1) * 100
    df['dom'] = df['Date'].dt.day  # day of month

    # Jours proches du 10 du mois (±3 jours)
    near_wasde = df[df['dom'].between(7, 13)]
    not_wasde  = df[~df['dom'].between(7, 13)]

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.hist(near_wasde['ret_1d'].dropna(), bins=60, alpha=0.6, label='Jours proches WASDE (±3j du 10)', density=True)
    ax.hist(not_wasde['ret_1d'].dropna(), bins=60, alpha=0.6, label='Autres jours', density=True)
    ax.set_title('Distribution des rendements journaliers : jours WASDE vs. autres jours')
    ax.set_xlabel('Rendement journalier (%)')
    ax.legend()
    ax.set_xlim(-8, 8)
    plt.tight_layout()
    plt.show()

    print(f"Volatilité jours WASDE : {near_wasde['ret_1d'].std():.3f}%")
    print(f"Volatilité autres jours : {not_wasde['ret_1d'].std():.3f}%")
else:
    print('Colonne prix non disponible pour cette analyse.')

## 6. Conclusions saisonnales

### Ce qu'on retient

1. **La saisonnalité existe** — les rendements en juillet/août diffèrent statistiquement du reste de l'année.

2. **Elle n'est pas stable d'une décennie à l'autre** — les crises (2012, 2022) peuvent inverser les patterns.

3. **La volatilité est maximale en été** (juillet-août) et minimale en hiver. Cela valide l'idée d'un CQR par saison.

4. **Les jours WASDE sont plus volatils** → l'information fondamentale USDA crée des sauts de prix.

### Ce que ça change pour notre modélisation

- On encode la saisonnalité par des termes `sin(2π·mois/12)` et `cos(2π·mois/12)` plutôt que des dummies
- On crée un indicateur `is_growing_season` (mai-septembre)
- On calibre le CQR séparément en été vs. hiver (prochain carnet)
- Le `baseline_seasonal_naive` qui beat les modèles ML à h=20/30j s'explique ici : la saisonnalité est un signal si fort qu'un modèle qui ne la capture pas sera battu

**→ Prochain carnet :** Facteurs fondamentaux et analyse WASDE.